In [1]:
# ============================================================
# NOTEBOOK — (A) BLEND SUBMISSION ESTERNE + (B) MEGA STACKING PIRAMIDALE (OOF)
# Obiettivo: avvicinarsi a LB top10 con:
#   A) robust aggregation su 4 submission esterne (q-quantile / rank / odds / trimmed)
#   B) stacking "piramidale" GM-style usando TUTTI i tuoi OOF+pred .npy (100+ modelli)
#   C) final blend: (B) + (A) con rank-blend a pesi piccoli
#
# OUTPUT in /kaggle/working:
#   - sub_ext_*.csv                    (candidati solo-esterni)
#   - sub_pyramid_internal.csv         (stacking interno)
#   - sub_final_internal_plus_ext*.csv (blend finale interno+esterni)
#   - pyramid_info.json
# ============================================================

import os, glob, json, gc, warnings, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import Ridge

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = "/kaggle/input/playground-series-s6e2"
OUTDIR   = "/kaggle/working"

SAMPLE = os.path.join(DATA_DIR, "sample_submission.csv")
TRAIN  = os.path.join(DATA_DIR, "train.csv")
TEST   = os.path.join(DATA_DIR, "test.csv")

# --- le 4 submission esterne che ti danno ~0.95398 con quantile(20%) ---
EXT_SUBS = [
    "/kaggle/input/the-best-solo-model-so-far-realmlp-lb-0-95397/submission.csv",
    "/kaggle/input/heart-01/submission.csv",
    "/kaggle/input/predicting-heart-disease-blend/submission.csv",
    "/kaggle/input/notebooks/harukiharada/realmlp-ext-target-stats-5-fold-cv/submission.csv",
]

# (opzionale) submission esterna extra singola che mi hai dato prima (anchor)
MAYBE_EXTRA_ANCHOR = "/kaggle/input/notebooks/azzamradman/0226-blend-the-blender/submission.csv"

# =========================
# HELPERS
# =========================
def rank01(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x)
    order = np.argsort(x, kind="mergesort")
    r = np.empty_like(order, dtype=np.float32)
    r[order] = np.arange(len(x), dtype=np.float32)
    denom = (len(x) - 1) if len(x) > 1 else 1.0
    return r / (denom + 1e-12)

def rank_matrix(P: np.ndarray) -> np.ndarray:
    R = np.empty_like(P, dtype=np.float32)
    for j in range(P.shape[1]):
        R[:, j] = rank01(P[:, j])
    return R

def odds_mean(P: np.ndarray, eps=1e-6) -> np.ndarray:
    Q = np.clip(P, eps, 1 - eps)
    L = np.log(Q/(1-Q))
    m = L.mean(axis=1)
    return (1/(1+np.exp(-m))).astype(np.float32)

def trimmed_mean(P: np.ndarray, trim=0.25) -> np.ndarray:
    S = np.sort(P, axis=1)
    lo = int(np.floor(trim * S.shape[1]))
    hi = int(np.ceil((1-trim) * S.shape[1]))
    hi = max(hi, lo+1)
    return S[:, lo:hi].mean(axis=1).astype(np.float32)

def weighted_quantile_rowwise(P: np.ndarray, w: np.ndarray, q: float) -> np.ndarray:
    w = np.asarray(w, dtype=np.float32)
    w = np.maximum(w, 0)
    w = w / (w.sum() + 1e-12)

    idx = np.argsort(P, axis=1)
    Ps = np.take_along_axis(P, idx, axis=1)
    Ws = np.take_along_axis(np.tile(w, (P.shape[0], 1)), idx, axis=1)

    cum = np.cumsum(Ws, axis=1)
    cutoff = q * cum[:, -1]
    pos = (cum >= cutoff[:, None]).argmax(axis=1)
    return Ps[np.arange(P.shape[0]), pos].astype(np.float32)

def save_sub(sample_df, target_col, pred, filename):
    out = sample_df.copy()
    out[target_col] = np.asarray(pred, dtype=np.float32)
    path = os.path.join(OUTDIR, filename)
    out.to_csv(path, index=False)
    print("✅ saved:", path)

def safe_load_npy(path):
    arr = np.load(path)
    arr = np.asarray(arr, dtype=np.float32)
    if np.isnan(arr).any():
        m = float(np.nanmedian(arr))
        arr = np.nan_to_num(arr, nan=m)
    return arr

def logit_clip(p, eps=1e-6):
    p = np.clip(p, eps, 1-eps)
    return np.log(p/(1-p))

def infer_family(s: str) -> str:
    t = s.lower()
    if "cat" in t: return "cat"
    if "dart" in t: return "dart"
    if "lgb" in t: return "lgb"
    if "xgb" in t: return "xgb"
    if "extra" in t or "et" in t or "extratrees" in t: return "et"
    if "nn" in t or "tabnet" in t or "mlp" in t or "realmlp" in t: return "nn"
    return "other"

def ridge_oof_pred(X, y, T, alphas=(0.2, 0.5, 1, 2, 5), n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    best = (-1.0, None, None, None)  # auc, alpha, oof, pte

    for a in alphas:
        oof = np.zeros(len(y), dtype=np.float32)
        pte = np.zeros(T.shape[0], dtype=np.float32)

        for fold, (tr, va) in enumerate(skf.split(X, y), 1):
            m = Ridge(alpha=float(a), random_state=seed + fold)
            m.fit(X[tr], y[tr])
            oof[va] = m.predict(X[va]).astype(np.float32)
            pte += m.predict(T).astype(np.float32) / n_splits

        oof = np.clip(oof, 0, 1)
        pte = np.clip(pte, 0, 1)
        auc = roc_auc_score(y, oof)

        if auc > best[0]:
            best = (float(auc), float(a), oof, pte)

    return best  # auc, alpha, oof, pte

# =========================
# LOAD BASE DATA
# =========================
train = pd.read_csv(TRAIN)
test  = pd.read_csv(TEST)
sam   = pd.read_csv(SAMPLE)

ID_COL = "id" if "id" in train.columns else train.columns[0]
TARGET_COL = next((c for c in ["Heart Disease","target","Target","heart_disease","HeartDisease"] if c in train.columns), None)
if TARGET_COL is None:
    raise ValueError("Target non trovato nel train.")
SUB_TARGET_COL = [c for c in sam.columns if c != ID_COL][0]

y_raw = train[TARGET_COL]
if y_raw.dtype == "object":
    y = y_raw.map({"Presence":1, "Absence":0}).astype(int).values
else:
    y = y_raw.astype(int).values

print(f"ID_COL={ID_COL} | TARGET_COL={TARGET_COL} | SUB_TARGET_COL={SUB_TARGET_COL}")
print("Train/Test:", train.shape, test.shape, "| Pos rate:", float(y.mean()))
print()

# ============================================================
# (A) BLEND ESTERNI — ROBUST AGGREGATION (molto forte su LB)
# ============================================================
print("=== (A) External blend candidates ===")

ext_paths = [p for p in EXT_SUBS if os.path.exists(p)]
if os.path.exists(MAYBE_EXTRA_ANCHOR):
    ext_paths.append(MAYBE_EXTRA_ANCHOR)
    print("✅ Added extra anchor:", MAYBE_EXTRA_ANCHOR)

if len(ext_paths) < 2:
    print("⚠️ Non ho abbastanza submission esterne trovate. Skipping (A).")
    P_ext = None
else:
    preds = []
    used = []
    for p in ext_paths:
        df = pd.read_csv(p)
        col = SUB_TARGET_COL if SUB_TARGET_COL in df.columns else [c for c in df.columns if c != ID_COL][0]
        df = df[[ID_COL, col]].rename(columns={col: SUB_TARGET_COL})
        df = sam[[ID_COL]].merge(df, on=ID_COL, how="left").sort_values(ID_COL)
        if df[SUB_TARGET_COL].isna().any():
            raise ValueError(f"NaN dopo merge: {p} (mismatch ID?)")
        preds.append(df[SUB_TARGET_COL].astype(np.float32).values)
        used.append(p)

    P_ext = np.stack(preds, axis=1)  # (n_test, k)
    k = P_ext.shape[1]
    print("Loaded external preds:", P_ext.shape)
    for u in used:
        print(" -", u)
    print()

    R_ext = rank_matrix(P_ext)

    # candidati come fanno i top: quantile su prob + quantile su rank + odds mean + trimmed mean
    for q in [0.15, 0.20, 0.25]:
        save_sub(sam, SUB_TARGET_COL, np.quantile(P_ext, q, axis=1), f"sub_ext_q{int(q*100):02d}_prob.csv")
        save_sub(sam, SUB_TARGET_COL, np.quantile(R_ext, q, axis=1), f"sub_ext_q{int(q*100):02d}_rank.csv")

    save_sub(sam, SUB_TARGET_COL, R_ext.mean(axis=1), "sub_ext_rank_mean.csv")
    save_sub(sam, SUB_TARGET_COL, odds_mean(P_ext), "sub_ext_odds_mean.csv")
    if k >= 4:
        save_sub(sam, SUB_TARGET_COL, trimmed_mean(P_ext, trim=0.25), "sub_ext_trimmed_mean.csv")

    # weighted rank quantile: dai più peso al 1° modello (spesso realmlp forte)
    if k == 4:
        w = np.array([0.45, 0.20, 0.20, 0.15], np.float32)
    elif k == 5:
        w = np.array([0.40, 0.20, 0.20, 0.10, 0.10], np.float32)
    else:
        w = np.ones(k, np.float32) / k

    save_sub(sam, SUB_TARGET_COL, weighted_quantile_rowwise(R_ext, w, q=0.20), "sub_ext_wq20_rank.csv")

print()

# ============================================================
# (B) MEGA STACKING PIRAMIDALE (serve OOF) — GM style
#   Livello 0: 100+ modelli (OOF/PTE)
#   Livello 1: Ridge(rank), Ridge(logit-z), robuste aggregazioni (rank mean/q20, odds)
#   Livello 2: Ridge finale su [L1 outputs + family avgs]
# ============================================================
print("=== (B) Pyramid stacking (OOF) ===")

# --- discovery robusto: ovunque in /kaggle/working e /kaggle/input ---
SEARCH_ROOTS = ["/kaggle/working", "/kaggle/input"]
oof_paths = []
for root in SEARCH_ROOTS:
    oof_paths += glob.glob(os.path.join(root, "**", "oof_*.npy"), recursive=True)

pairs = []
for oofp in oof_paths:
    name = os.path.basename(oofp)[4:-4]
    predp = os.path.join(os.path.dirname(oofp), f"pred_{name}_test.npy")
    if os.path.exists(predp):
        pairs.append((oofp, predp))

print("Found paired (oof,pred):", len(pairs))

models, families = [], []
OOF_list, PTE_list = [], []

for oofp, predp in pairs:
    oof = safe_load_npy(oofp)
    pte = safe_load_npy(predp)

    if len(oof) != len(train) or len(pte) != len(test):
        continue
    if float(np.std(oof)) < 1e-10:
        continue

    folder = os.path.basename(os.path.dirname(oofp))
    name = os.path.basename(oofp)[4:-4]
    key = f"{folder}__{name}"

    models.append(key)
    families.append(infer_family(oofp))
    OOF_list.append(oof)
    PTE_list.append(pte)

if len(models) < 10:
    print("⚠️ Pochi modelli OOF trovati. Skipping (B).")
    internal_pred = None
else:
    OOF = np.vstack(OOF_list).T.astype(np.float32)  # (n_train, m)
    PTE = np.vstack(PTE_list).T.astype(np.float32)  # (n_test, m)

    # fix inversion + score
    aucs = np.zeros(OOF.shape[1], np.float64)
    for j in range(OOF.shape[1]):
        a = roc_auc_score(y, OOF[:, j])
        if a < 0.5:
            OOF[:, j] = 1.0 - OOF[:, j]
            PTE[:, j] = 1.0 - PTE[:, j]
            a = 1.0 - a
        aucs[j] = a

    meta = pd.DataFrame({"model": models, "family": families, "cv_auc": aucs}).sort_values("cv_auc", ascending=False).reset_index(drop=True)
    print("Loaded OOF models:", len(meta))
    print("Best single OOF:", meta.iloc[0]["model"], "|", float(meta.iloc[0]["cv_auc"]))
    print()

    # filtro debole (tienilo non troppo aggressivo)
    MIN_AUC_KEEP = 0.90
    meta = meta[meta["cv_auc"] >= MIN_AUC_KEEP].reset_index(drop=True)
    idx_keep = [models.index(m) for m in meta["model"].tolist()]
    OOF = OOF[:, idx_keep]
    PTE = PTE[:, idx_keep]
    models = meta["model"].tolist()
    families = meta["family"].tolist()
    aucs = meta["cv_auc"].values.astype(np.float64)
    print("After MIN_AUC_KEEP:", len(models))

    # corr-prune rapido (NO full corr matrix): greedy su subset
    SUB_N = 120_000
    rng = np.random.RandomState(SEED)

    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_pos = int(SUB_N * y.mean())
    n_neg = SUB_N - n_pos
    sub_idx = np.concatenate([
        rng.choice(pos_idx, size=min(n_pos, len(pos_idx)), replace=False),
        rng.choice(neg_idx, size=min(n_neg, len(neg_idx)), replace=False),
    ])
    rng.shuffle(sub_idx)

    Xsub = OOF[sub_idx]
    order = np.argsort(-aucs)
    keep = []
    CORR_TH = 0.9993

    def corr1(a, b):
        a = a - a.mean()
        b = b - b.mean()
        return float((a*b).mean() / (np.sqrt((a*a).mean())*np.sqrt((b*b).mean()) + 1e-12))

    for i in order:
        if len(keep) == 0:
            keep.append(i)
            continue
        ok = True
        cand = Xsub[:, i]
        for kidx in keep:
            if corr1(cand, Xsub[:, kidx]) > CORR_TH:
                ok = False
                break
        if ok:
            keep.append(i)

    keep = np.array(keep, dtype=int)
    OOF = OOF[:, keep]
    PTE = PTE[:, keep]
    models = [models[i] for i in keep]
    families = [families[i] for i in keep]
    aucs = aucs[keep]
    print(f"After CORR prune>{CORR_TH}: kept {len(models)} models")
    print()

    # --- Level 0 transforms ---
    # Rank
    print("Rank transform...")
    OOF_rank = np.empty_like(OOF, dtype=np.float32)
    PTE_rank = np.empty_like(PTE, dtype=np.float32)
    for j in range(OOF.shape[1]):
        OOF_rank[:, j] = rank01(OOF[:, j])
        PTE_rank[:, j] = rank01(PTE[:, j])

    # Logit-Z
    print("Logit-z transform...")
    eps = 1e-6
    OOF_log = logit_clip(OOF.astype(np.float64), eps=eps).astype(np.float32)
    PTE_log = logit_clip(PTE.astype(np.float64), eps=eps).astype(np.float32)
    mu = OOF_log.mean(axis=0, dtype=np.float64)
    sd = OOF_log.std(axis=0, dtype=np.float64)
    sd[sd < 1e-6] = 1.0
    OOF_logz = ((OOF_log - mu) / sd).astype(np.float32)
    PTE_logz = ((PTE_log - mu) / sd).astype(np.float32)
    del OOF_log, PTE_log
    gc.collect()

    # Family averages (su rank e su logit-z)
    fams = np.array(families)
    uniq_fams = sorted(pd.unique(fams))
    fam_rank_oof, fam_rank_pte, fam_log_oof, fam_log_pte, fam_names = [], [], [], [], []

    for fam in uniq_fams:
        cols = np.where(fams == fam)[0]
        if len(cols) == 0: 
            continue
        fam_names.append(f"famavg_{fam}")
        fam_rank_oof.append(OOF_rank[:, cols].mean(axis=1))
        fam_rank_pte.append(PTE_rank[:, cols].mean(axis=1))
        fam_log_oof.append(OOF_logz[:, cols].mean(axis=1))
        fam_log_pte.append(PTE_logz[:, cols].mean(axis=1))

    fam_rank_oof = np.vstack(fam_rank_oof).T.astype(np.float32) if len(fam_rank_oof) else np.zeros((len(train),0), np.float32)
    fam_rank_pte = np.vstack(fam_rank_pte).T.astype(np.float32) if len(fam_rank_pte) else np.zeros((len(test),0), np.float32)
    fam_log_oof  = np.vstack(fam_log_oof).T.astype(np.float32)  if len(fam_log_oof)  else np.zeros((len(train),0), np.float32)
    fam_log_pte  = np.vstack(fam_log_pte).T.astype(np.float32)  if len(fam_log_pte)  else np.zeros((len(test),0), np.float32)

    # --- Level 1 models (Ridge rank + Ridge logit-z) ---
    alphas = (0.1, 0.2, 0.5, 1, 2, 5)

    print("L1: Ridge on rank meta...")
    X1_rank = np.hstack([OOF_rank, fam_rank_oof]).astype(np.float32)
    T1_rank = np.hstack([PTE_rank, fam_rank_pte]).astype(np.float32)
    auc_r1, a_r1, oof_r1, pte_r1 = ridge_oof_pred(X1_rank, y, T1_rank, alphas=alphas, n_splits=5, seed=SEED)
    print("  -> best:", auc_r1, "alpha:", a_r1)

    print("L1: Ridge on logit-z meta...")
    X1_log = np.hstack([OOF_logz, fam_log_oof]).astype(np.float32)
    T1_log = np.hstack([PTE_logz, fam_log_pte]).astype(np.float32)
    auc_l1, a_l1, oof_l1, pte_l1 = ridge_oof_pred(X1_log, y, T1_log, alphas=alphas, n_splits=5, seed=SEED)
    print("  -> best:", auc_l1, "alpha:", a_l1)
    print()

    # --- Level 1 robust aggregations (sul TRAIN usiamo OOF, sul TEST usiamo PTE) ---
    # (sono “modelli” gratis e spesso aiutano parecchio)
    oof_rank_mean = OOF_rank.mean(axis=1).astype(np.float32)
    pte_rank_mean = PTE_rank.mean(axis=1).astype(np.float32)

    oof_rank_q20 = np.quantile(OOF_rank, 0.20, axis=1).astype(np.float32)
    pte_rank_q20 = np.quantile(PTE_rank, 0.20, axis=1).astype(np.float32)

    oof_odds = odds_mean(OOF)
    pte_odds = odds_mean(PTE)

    # --- Level 2 (Piramide): Ridge su outputs L1 + fam avgs (pochi feature = super stabile) ---
    X2 = np.vstack([
        oof_r1, oof_l1,
        oof_rank_mean, oof_rank_q20,
        oof_odds
    ]).T.astype(np.float32)

    T2 = np.vstack([
        pte_r1, pte_l1,
        pte_rank_mean, pte_rank_q20,
        pte_odds
    ]).T.astype(np.float32)

    print("L2: final Ridge on pyramid outputs...")
    auc2, a2, oof2, pte2 = ridge_oof_pred(X2, y, T2, alphas=(0.05, 0.1, 0.2, 0.5, 1, 2), n_splits=5, seed=SEED)
    print("  -> L2 OOF AUC:", auc2, "alpha:", a2)

    internal_pred = np.clip(pte2, 0, 1).astype(np.float32)
    save_sub(sam, SUB_TARGET_COL, internal_pred, "sub_pyramid_internal.csv")

    info = {
        "n_models_loaded": int(len(models)),
        "min_auc_keep": float(MIN_AUC_KEEP),
        "corr_th": float(CORR_TH),
        "L1_ridge_rank_auc": float(auc_r1),
        "L1_ridge_rank_alpha": float(a_r1),
        "L1_ridge_logit_auc": float(auc_l1),
        "L1_ridge_logit_alpha": float(a_l1),
        "L2_auc": float(auc2),
        "L2_alpha": float(a2),
        "fam_names": fam_names,
        "pyramid_features": ["ridge_rank","ridge_logit","rank_mean","rank_q20","odds_mean"],
    }
    with open(os.path.join(OUTDIR, "pyramid_info.json"), "w") as f:
        json.dump(info, f, indent=2)
    print("Saved:", os.path.join(OUTDIR, "pyramid_info.json"))
    print()

# ============================================================
# (C) FINAL BLEND: piramide interna + esterno (rank-blend pesi piccoli)
# ============================================================
print("=== (C) Final blends (internal pyramid + external) ===")

# scegli come esterno migliore: per default usa l’output più forte tipico "sub_ext_q20_prob.csv"
best_ext_candidate = os.path.join(OUTDIR, "sub_ext_q20_prob.csv")
if not os.path.exists(best_ext_candidate):
    # fallback: wq20_rank
    best_ext_candidate = os.path.join(OUTDIR, "sub_ext_wq20_rank.csv")

if internal_pred is None:
    print("⚠️ Non ho internal_pred (piramide). Salvo solo i candidati esterni già creati.")
else:
    if os.path.exists(best_ext_candidate):
        ext_df = pd.read_csv(best_ext_candidate)
        ext_pred = ext_df[SUB_TARGET_COL].astype(np.float32).values

        int_rank = rank01(internal_pred)
        ext_rank = rank01(ext_pred)

        # pesi piccoli: di solito sono i più “safe” per non distruggere PB
        for w in [0.01, 0.02, 0.03, 0.05, 0.08]:
            final = (1.0 - w) * int_rank + w * ext_rank
            save_sub(sam, SUB_TARGET_COL, final, f"sub_final_internal_plus_ext_w{str(w).replace('.','p')}.csv")

        print("\nConsiglio submit ordine:")
        print("1) /kaggle/working/sub_pyramid_internal.csv")
        print("2) /kaggle/working/sub_final_internal_plus_ext_w0p02.csv")
        print("3) /kaggle/working/sub_final_internal_plus_ext_w0p05.csv")
    else:
        print("⚠️ Non trovo best_ext_candidate:", best_ext_candidate)

print("\nDONE.")


ID_COL=id | TARGET_COL=Heart Disease | SUB_TARGET_COL=Heart Disease
Train/Test: (630000, 15) (270000, 14) | Pos rate: 0.44833968253968254

=== (A) External blend candidates ===
✅ Added extra anchor: /kaggle/input/notebooks/azzamradman/0226-blend-the-blender/submission.csv
Loaded external preds: (270000, 2)
 - /kaggle/input/notebooks/harukiharada/realmlp-ext-target-stats-5-fold-cv/submission.csv
 - /kaggle/input/notebooks/azzamradman/0226-blend-the-blender/submission.csv

✅ saved: /kaggle/working/sub_ext_q15_prob.csv
✅ saved: /kaggle/working/sub_ext_q15_rank.csv
✅ saved: /kaggle/working/sub_ext_q20_prob.csv
✅ saved: /kaggle/working/sub_ext_q20_rank.csv
✅ saved: /kaggle/working/sub_ext_q25_prob.csv
✅ saved: /kaggle/working/sub_ext_q25_rank.csv
✅ saved: /kaggle/working/sub_ext_rank_mean.csv
✅ saved: /kaggle/working/sub_ext_odds_mean.csv
✅ saved: /kaggle/working/sub_ext_wq20_rank.csv

=== (B) Pyramid stacking (OOF) ===
Found paired (oof,pred): 103
Loaded OOF models: 103
Best single OOF: mo